# 05 · Hybrid Refinement: Rescuing Decoys
### *Combining Physics-Based Minimization with Experimental Gradients*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/elkins-lab/diff-biophys/blob/main/examples/interactive_tutorials/05_hybrid_refinement_pdb.ipynb)

---

**What you will learn:**
1. How to load and manipulate protein structures from PDB files using **Biotite**.
2. How to use an energy-minimized structure from **synth-pdb** as a starting point.
3. How to perform "Hybrid Refinement": using experimental gradients to pull a physically-plausible decoy toward the native state.

**Time:** ~25 minutes

## 0 · Setup

Install the ecosystem libraries.

In [ ]:
# Install diff-biophys, biotite, and optimization tools
%pip install -q diff-biophys==0.1.6 biotite optax matplotlib synth-pdb synth-nmr

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import optax
import biotite.structure as stripe
import biotite.structure.io.pdb as pdb
import numpy as np
import os

from diff_biophys.geometry.nerf import chain_nerf
from diff_biophys.geometry.torsions import compute_dihedrals
from diff_biophys.nmr.rdc import calculate_rdc, calculate_q_factor

jax.config.update("jax_enable_x64", False)

## 1 · Generate the Decoy with synth-pdb

We'll use `synth-pdb` to generate two structures:
1. A **Target** alpha-helix (the "Ground Truth").
2. A **Minimized Decoy** (the "Starting Point") which is a random coil that has been energy-minimized using OpenMM to ensure perfect sterics.

In [ ]:
!synth-pdb --sequence AAAAAAAAAA --conformation alpha --output target.pdb
!synth-pdb --sequence AAAAAAAAAA --conformation random --minimize --output initial_minimized.pdb

## 2 · Extract Backbone Coordinates

We'll use `biotite` to read the PDB files and extract the N-CA-C backbone atoms.

In [ ]:
def get_backbone_coords(pdb_path):
    pdb_file = pdb.PDBFile.read(pdb_path)
    struct = pdb_file.get_structure(model=1)
    mask = (struct.atom_name == "N") | (struct.atom_name == "CA") | (struct.atom_name == "C")
    backbone = struct[mask]
    return jnp.array(backbone.coord, dtype=jnp.float32)

target_coords = get_backbone_coords("target.pdb")
initial_coords = get_backbone_coords("initial_minimized.pdb")

N_RES = 10
print(f"Loaded structures with {N_RES} residues.")

## 3 · Setup Experimental Constraints (RDCs)

We'll simulate Residual Dipolar Couplings (RDCs) for the target structure. Our goal is to refine the minimized decoy to match these couplings.

In [ ]:
def get_peptide_bond_vectors(coords):
    c_atoms = coords[2::3][:-1]
    n_atoms = coords[3::3]
    vecs = n_atoms - c_atoms
    return vecs / jnp.linalg.norm(vecs, axis=-1, keepdims=True)

target_vectors = get_peptide_bond_vectors(target_coords)
target_rdcs = calculate_rdc(target_vectors, da=10.0, r=0.1)

print(f"Target RDCs generated: {target_rdcs[:3]}...")

## 4 · The Refinement Loop

We'll use a torsional model (NeRF) to parameterize the structure and `optax` to minimize the RDC Q-factor.

In [ ]:
# Helper to extract phi/psi from coords
def compute_phi_psi(coords):
    d = compute_dihedrals(coords)
    psi = jnp.concatenate([d[0::3], jnp.array([0.0])])
    phi = jnp.concatenate([jnp.array([0.0]), d[2::3]])
    return phi, psi

init_phi, init_psi = compute_phi_psi(initial_coords)

# Ideal backbone parameters
CA_C_LENGTH, C_N_LENGTH, N_CA_LENGTH = 1.525, 1.329, 1.459
CA_C_N, C_N_CA, N_CA_C = jnp.radians(116.2), jnp.radians(121.7), jnp.radians(111.2)
lengths = jnp.array([C_N_LENGTH, N_CA_LENGTH, CA_C_LENGTH] * (N_RES - 1))
angles = jnp.array([CA_C_N, C_N_CA, N_CA_C] * (N_RES - 1))

def build_structure(phi, psi):
    omega = jnp.full((N_RES - 1,), jnp.pi)
    d = jnp.stack([psi[:-1], omega, phi[1:]], axis=1).ravel()
    return chain_nerf(initial_coords[:3], lengths, angles, d)

def loss_fn(params):
    phi, psi = params
    coords = build_structure(phi, psi)
    vectors = get_peptide_bond_vectors(coords)
    rdcs = calculate_rdc(vectors, da=10.0, r=0.1)
    return calculate_q_factor(rdcs, target_rdcs)

optimizer = optax.adam(learning_rate=0.02)
params = (init_phi, init_psi)
opt_state = optimizer.init(params)

@jax.jit
def step(params, opt_state):
    loss, grads = jax.value_and_grad(loss_fn)(params)
    updates, opt_state = optimizer.update(grads, opt_state)
    params = optax.apply_updates(params, updates)
    return params, opt_state, loss

print("Refining...")
losses = []
for i in range(151):
    params, opt_state, loss = step(params, opt_state)
    losses.append(loss)
    if i % 50 == 0:
        print(f"Step {i:03d} | Q-factor: {loss:.4f}")

## 5 · Visualise the Improvement

Finally, we'll plot the refinement trajectory and see the structural alignment.

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(losses, lw=2, color='darkblue')
plt.title("Refinement Trajectory (RDC Q-factor)")
plt.xlabel("Iteration")
plt.ylabel("Q-factor")
plt.grid(alpha=0.3)
plt.show()

final_coords = build_structure(*params)

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
ax.plot(target_coords[:, 0], target_coords[:, 1], target_coords[:, 2], 'k--', alpha=0.3, label="Target")
ax.plot(initial_coords[:, 0], initial_coords[:, 1], initial_coords[:, 2], 'b-', alpha=0.3, label="Initial Decoy")
ax.plot(final_coords[:, 0], final_coords[:, 1], final_coords[:, 2], 'r-', lw=3, label="Refined Result")
ax.legend()
ax.set_title("Structural Rescue via Experimental Gradients")
plt.show()